# 📊 Prometheus & Grafana Monitoring Dashboard
Dieses Notebook dient als Management-Konsole für das Infrastruktur- und Modell-Monitoring deines Fraud-Detection-Projekts.

In [1]:
import os
import requests
from IPython.display import display, HTML

# ==========================================
# SCHRITT 1: Konfiguration & Endpunkte
# ==========================================
print("=== Prometheus & Grafana Infrastruktur-Check ===")

PROMETHEUS_URL = "http://prometheus:9090"
GRAFANA_URL = "http://grafana:3000"

# ==========================================
# SCHRITT 2: Status-Prüfung (Health Checks)
# ==========================================
def check_service(name, url):
    try:
        response = requests.get(url, timeout=3)
        if response.status_code < 400:
            print(f"✅ {name} ist erreichbar unter: {url}")
            return True
        else:
            print(f"⚠️ {name} antwortet mit Statuscode: {response.status_code}")
            return False
    except requests.ConnectionError:
        print(f"❌ {name} ist NICHT erreichbar ({url}). Läuft der Dienst?")
        return False

print("\n--- Überprüfe laufende Services ---")
prom_status = check_service("Prometheus", PROMETHEUS_URL)
grafana_status = check_service("Grafana", GRAFANA_URL)

# ==========================================
# SCHRITT 3: GPU-Metriken Check (DCGM Exporter)
# ==========================================
if prom_status:
    print("\n--- Überprüfe Prometheus GPU-Metriken ---")
    try:
        query_url = f"{PROMETHEUS_URL}/api/v1/query?query=up"
        res = requests.get(query_url, timeout=3).json()
        
        targets = res.get("data", {}).get("result", [])
        print(f"📊 Aktive Prometheus-Targets gefunden: {len(targets)}")
        for target in targets:
            job = target.get("metric", {}).get("job", "unknown")
            health = target.get("value", ["", "unknown"])[1]
            print(f"   - Job '{job}': Status {health}")
    except Exception as e:
        print(f"⚠️ Konnte Prometheus-Metriken nicht abfragen: {e}")

# ==========================================
# SCHRITT 4: Nützliche Dashboards verlinken
# ==========================================
print("\n--- Dashboard-Verknüpfungen ---")
display(HTML(f"""
<div style="padding: 15px; border: 1px solid #4CAF50; border-radius: 5px; background-color: #f9f9f9;">
    <h3>🔗 Schnellzugriff auf eure Monitoring-Oberflächen</h3>
    <ul>
        <li><b>Grafana Web-UI:</b> <a href="{GRAFANA_URL}" target="_blank">{GRAFANA_URL}</a> (Ideal für Visualisierung von Loss, GPU-Auslastung & VRAM)</li>
        <li><b>Prometheus Web-UI:</b> <a href="{PROMETHEUS_URL}" target="_blank">{PROMETHEUS_URL}</a> (Für direkte PromQL-Abfragen)</li>
    </ul>
    <p><i>Tipp: Falls die Links nicht öffnen, stelle sicher, dass die Ports via SSH-Port-Forwarding (z.B. <code>-L 3000:localhost:3000 -L 9090:localhost:9090</code>) weitergeleitet wurden.</i></p>
</div>
"""))

=== Prometheus & Grafana Infrastruktur-Check ===

--- Überprüfe laufende Services ---
✅ Prometheus ist erreichbar unter: http://prometheus:9090
✅ Grafana ist erreichbar unter: http://grafana:3000

--- Überprüfe Prometheus GPU-Metriken ---
📊 Aktive Prometheus-Targets gefunden: 2
   - Job 'prometheus': Status 1
   - Job 'gpu-metrics': Status 1

--- Dashboard-Verknüpfungen ---
